# EcoMind AI
### AI-Powered Solar Intelligence & Decision Support

**Veritas AI International High School Hackathon 2026 — Climate + AI**  
**Primary research location:** Dhaka, Bangladesh  
**Location experience:** Global city search; formal scientific evaluation is Dhaka-only.

This notebook predicts **next-day daily shortwave solar radiation** using current-day weather and solar conditions.

**Scientific scope:** Solar radiation is not the same as electricity generated by a solar panel. Electricity output also depends on panel capacity, tilt, azimuth, efficiency, shading, and system losses.

Competition requirements: chronological train/test split (no shuffle), persistence baseline, MAE comparison, and a clear real-world decision supported by the forecast.

## 1. Install libraries
Run from top to bottom in Google Colab.

In [ ]:
!pip -q install gradio==5.49.1 requests pandas numpy scikit-learn matplotlib plotly

In [ ]:
import warnings, requests
from datetime import date, timedelta
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from IPython.display import display
warnings.filterwarnings("ignore")

PROJECT_NAME = "EcoMind AI"
HISTORICAL_URL = "https://archive-api.open-meteo.com/v1/archive"
GEOCODING_URL = "https://geocoding-api.open-meteo.com/v1/search"
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"

DAILY_VARIABLES = [
    "shortwave_radiation_sum", "temperature_2m_mean", "temperature_2m_max",
    "temperature_2m_min", "precipitation_sum", "cloud_cover_mean",
    "wind_speed_10m_max", "relative_humidity_2m_mean",
    "daylight_duration", "sunshine_duration"
]
FEATURES = DAILY_VARIABLES + ["month_sin", "month_cos"]
TARGET = "target_next_day_radiation"
print(PROJECT_NAME, "— environment ready")

## 2. Find coordinates
The primary evaluation location is Dhaka. The same geocoder powers the global city explorer.

In [ ]:
def geocode_city(query):
    r = requests.get(GEOCODING_URL, params={
        "name": query, "count": 5, "language": "en", "format": "json"
    }, timeout=30)
    r.raise_for_status()
    hits = r.json().get("results", [])
    if not hits:
        raise ValueError(f"No location found for: {query}")
    p = hits[0]
    return {
        "name": p.get("name", query), "country": p.get("country", ""),
        "latitude": float(p["latitude"]), "longitude": float(p["longitude"]),
        "timezone": p.get("timezone", "auto")
    }

dhaka = geocode_city("Dhaka, Bangladesh")
dhaka

## 3. Download Dhaka historical data
We stop seven days before today to reduce incomplete recent-day records.

In [ ]:
def fetch_historical_daily(latitude, longitude, start_date="2015-01-01",
                           end_date=None, timezone="auto"):
    if end_date is None:
        end_date = (date.today() - timedelta(days=7)).isoformat()
    params = {
        "latitude": latitude, "longitude": longitude,
        "start_date": start_date, "end_date": end_date,
        "daily": ",".join(DAILY_VARIABLES), "timezone": timezone or "auto"
    }
    r = requests.get(HISTORICAL_URL, params=params, timeout=90)
    r.raise_for_status()
    payload = r.json()
    if "daily" not in payload:
        raise ValueError(f"No daily data returned: {payload}")
    df = pd.DataFrame(payload["daily"])
    df["time"] = pd.to_datetime(df["time"])
    return df.rename(columns={"time": "date"}).sort_values("date").reset_index(drop=True)

raw_dhaka = fetch_historical_daily(
    dhaka["latitude"], dhaka["longitude"], timezone=dhaka["timezone"]
)
print("Rows:", len(raw_dhaka), "| Period:",
      raw_dhaka.date.min().date(), "to", raw_dhaka.date.max().date())
display(raw_dhaka.tail())

## 4. Build next-day learning examples
Inputs are from day t; the target is radiation on day t+1. No future-day features are used.

In [ ]:
def make_learning_table(daily):
    df = daily.copy().sort_values("date").reset_index(drop=True)
    month = df["date"].dt.month
    df["month_sin"] = np.sin(2 * np.pi * month / 12)
    df["month_cos"] = np.cos(2 * np.pi * month / 12)
    df[TARGET] = df["shortwave_radiation_sum"].shift(-1)
    df["target_date"] = df["date"].shift(-1)
    return (df[FEATURES + [TARGET, "date", "target_date"]]
            .replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True))

learning_dhaka = make_learning_table(raw_dhaka)
if len(learning_dhaka) < 500:
    raise RuntimeError("Not enough historical rows. Check the API response.")
print("Learning examples:", len(learning_dhaka))
display(learning_dhaka[["date", "target_date",
                        "shortwave_radiation_sum", TARGET]].tail())

## 5. Chronological training and testing
Earliest 80% for training; latest 20% for testing. No random shuffle. Persistence predicts tomorrow = today.

In [ ]:
split = int(len(learning_dhaka) * 0.80)
train_df = learning_dhaka.iloc[:split].copy()
test_df = learning_dhaka.iloc[split:].copy()

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_test, y_test = test_df[FEATURES], test_df[TARGET]

model = RandomForestRegressor(
    n_estimators=350, min_samples_leaf=3, max_features=0.9,
    random_state=42, n_jobs=-1
)
model.fit(X_train, y_train)

ai_pred = np.clip(model.predict(X_test), 0, None)
baseline_pred = np.clip(test_df["shortwave_radiation_sum"].to_numpy(), 0, None)

ai_mae = mean_absolute_error(y_test, ai_pred)
baseline_mae = mean_absolute_error(y_test, baseline_pred)
results = pd.DataFrame({
    "Approach": ["EcoMind AI (Random Forest)", "Persistence baseline"],
    "MAE (MJ/m²/day)": [ai_mae, baseline_mae],
    "RMSE (MJ/m²/day)": [
        np.sqrt(mean_squared_error(y_test, ai_pred)),
        np.sqrt(mean_squared_error(y_test, baseline_pred))
    ]
})
print("Train examples:", len(train_df), "| Test examples:", len(test_df))
print("Held-out target dates:", test_df.target_date.min().date(),
      "to", test_df.target_date.max().date())
display(results)

## 6. Evaluate and visualise
Lower MAE is better. If the model does not beat persistence, report that honestly.

In [ ]:
plot_df = pd.DataFrame({
    "Date": test_df["target_date"].to_numpy(),
    "Actual": y_test.to_numpy(),
    "EcoMind AI": ai_pred,
    "Persistence": baseline_pred
}).melt(id_vars="Date", var_name="Series", value_name="Radiation (MJ/m²/day)")
px.line(plot_df, x="Date", y="Radiation (MJ/m²/day)", color="Series",
        title="Dhaka — held-out next-day solar radiation").show()

improvement = (baseline_mae - ai_mae) / baseline_mae * 100 if baseline_mae else 0
print(f"MAE change vs persistence: {improvement:.2f}% "
      "(positive means EcoMind AI has lower MAE)")

## 7. Explainable AI
Feature importance shows how the Random Forest used each feature; it does not establish causation.

In [ ]:
importance = pd.DataFrame({
    "Feature": FEATURES, "Importance": model.feature_importances_
}).sort_values("Importance", ascending=True)
px.bar(importance, x="Importance", y="Feature", orientation="h",
       title="EcoMind AI — feature importance").show()

## 8. Decision-support bands
Bands are derived from the training targets only. They are historical-relative categories, not universal engineering thresholds.

In [ ]:
q33, q67 = train_df[TARGET].quantile([0.33, 0.67]).tolist()

def decision_note(pred):
    if pred < q33:
        return ("Lower than the historical Dhaka range",
                "A solar-site or battery operator could review backup and charging plans.")
    if pred > q67:
        return ("Higher than the historical Dhaka range",
                "A solar-site operator could plan for comparatively favourable solar conditions.")
    return ("Within the middle historical range",
            "An operator could follow the normal operating plan while monitoring updated weather.")

print(f"Training-derived bands: low < {q33:.2f}; "
      f"middle {q33:.2f}–{q67:.2f}; high > {q67:.2f} MJ/m²/day")

## 9. Global City Explorer
For a selected city, today's forecast weather/radiation is passed to the Dhaka-trained model to predict tomorrow. Other-city results are exploratory, not globally validated.

In [ ]:
def fetch_today_features(latitude, longitude, timezone="auto"):
    params = {
        "latitude": latitude, "longitude": longitude,
        "daily": ",".join(DAILY_VARIABLES),
        "forecast_days": 2, "timezone": timezone or "auto"
    }
    r = requests.get(FORECAST_URL, params=params, timeout=60)
    r.raise_for_status()
    daily = pd.DataFrame(r.json()["daily"])
    daily["time"] = pd.to_datetime(daily["time"])
    daily = daily.sort_values("time").reset_index(drop=True)
    if len(daily) < 2:
        raise ValueError("Forecast API returned fewer than two days.")
    row = daily.iloc[0]
    month = row["time"].month
    values = {f: float(row[f]) for f in DAILY_VARIABLES}
    values["month_sin"] = float(np.sin(2 * np.pi * month / 12))
    values["month_cos"] = float(np.cos(2 * np.pi * month / 12))
    return pd.DataFrame([values])[FEATURES], row["time"].date(), daily.iloc[1]["time"].date()

def forecast_city(query):
    place = geocode_city(query)
    x, input_date, target_date = fetch_today_features(
        place["latitude"], place["longitude"], place["timezone"]
    )
    prediction = float(np.clip(model.predict(x)[0], 0, None))
    band, advice = decision_note(prediction)
    return place, input_date, target_date, prediction, band, advice

forecast_city("Dhaka, Bangladesh")

## 10. Interactive dashboard
Search by city and country, e.g. `Dhaka, Bangladesh` or `Nairobi, Kenya`.

In [ ]:
import gradio as gr

def dashboard_predict(query):
    try:
        place, input_date, target_date, prediction, band, advice = forecast_city(query)
        return (
            f"### {place['name']}, {place['country']}\n"
            f"Coordinates: {place['latitude']:.3f}, {place['longitude']:.3f}\n\n"
            f"**Weather input date:** {input_date}\n\n"
            f"## {prediction:.2f} MJ/m²/day\n"
            f"**Forecast target date:** {target_date}\n\n"
            f"**Historical-relative category:** {band}\n\n"
            f"**Decision context:** {advice}\n\n"
            "*Exploratory result: model evaluation is Dhaka-only.*"
        )
    except Exception as exc:
        return f"Could not create forecast: {type(exc).__name__}: {exc}"

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🌍 EcoMind AI\n### AI-Powered Solar Intelligence & Decision Support")
    gr.Markdown("Explore next-day solar radiation forecasts for cities around the world.")
    city = gr.Textbox(label="Search city and country", value="Dhaka, Bangladesh",
                      placeholder="Example: Dhaka, Bangladesh")
    run_btn = gr.Button("Generate next-day forecast", variant="primary")
    output = gr.Markdown()
    run_btn.click(dashboard_predict, inputs=city, outputs=output)
    gr.Markdown(
        "---\n**Research note:** formal evaluation uses a chronological Dhaka test period "
        "and persistence comparison. Other-city outputs are exploratory.\n\n"
        "**Data:** Open-Meteo historical reanalysis and weather forecasts."
    )

demo.launch(share=True)

## 11. Sources, limitations, and submission checks

### Data sources
- Open-Meteo Historical Weather API: https://open-meteo.com/en/docs/historical-weather-api
- Open-Meteo Forecast API: https://open-meteo.com/en/docs
- Open-Meteo Geocoding API: https://open-meteo.com/en/docs/geocoding-api

### Limitations to disclose
1. Target is daily shortwave radiation, not electricity generation.
2. Historical values are reanalysis estimates and may differ from site sensor measurements.
3. Formal evaluation is Dhaka-only.
4. Global city forecasts are exploratory; a Dhaka-trained model may not generalise to other climates.
5. Feature importance is not causal explanation.

### Before submitting
- Run every cell from top to bottom.
- Record the actual training and test date ranges and measured MAE values.
- Do not invent or manually edit metrics.
- Explain the model and its limitations in your own words.
- Disclose AI assistance according to the event rules.